# 26. Unsupervised Learning: DBSCAN (Density-Based Spatial Clustering)

## Algorithm Category
**Type**: Unsupervised Learning - Clustering  
**Complexity**: Medium  
**Use Case**: Density-based clustering that finds clusters of arbitrary shape and handles noise

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand DBSCAN algorithm and density-based clustering concepts
- Implement DBSCAN clustering
- Understand core points, border points, and noise points
- Tune hyperparameters (eps, min_samples)
- Handle noise/outliers automatically
- Apply DBSCAN to real-world problems

## Historical Context

DBSCAN was developed by Ester, Kriegel, Sander, and Xu in 1996:
- Ester, M., et al. (1996): "A density-based algorithm for discovering clusters in large spatial databases"
- Handles clusters of arbitrary shape
- Automatically identifies noise points

**Key Papers/References:**
- Ester, M., et al. (1996). "A density-based algorithm for discovering clusters in large spatial databases"

## When to Use DBSCAN

DBSCAN is appropriate when:
- You don't know the number of clusters
- Clusters have arbitrary shapes (non-spherical)
- You need to identify noise/outliers
- Clusters have varying densities
- You want to handle outliers automatically
- Working with spatial data

## Theory & Mechanics

### Mathematical Foundation

DBSCAN groups points that are closely packed together (density-connected).

**Key Concepts:**

1. **eps (ε)**: Maximum distance between two samples to be considered neighbors
2. **min_samples**: Minimum number of samples in a neighborhood to form a core point
3. **Core Point**: Point with at least min_samples neighbors within eps distance
4. **Border Point**: Point that is reachable from a core point but has fewer than min_samples neighbors
5. **Noise Point**: Point that is neither core nor border

**Algorithm Steps:**

1. **Initialize**: Mark all points as unvisited
2. **Select**: Pick an unvisited point p
3. **Check**: If p has at least min_samples neighbors within eps:
   - Create new cluster C
   - Add p to C
   - Mark p as visited
   - Add all density-reachable points to C
4. **Repeat**: Continue until all points visited

**Density-Reachable:**
- Point q is density-reachable from p if there's a chain of points where each is within eps of the next
- All points in a cluster are density-connected

### How It Works

1. **Find Core Points**: Points with enough neighbors
2. **Form Clusters**: Connect density-reachable core points
3. **Add Border Points**: Assign border points to nearest cluster
4. **Mark Noise**: Remaining points are noise/outliers

### Key Hyperparameters

- **eps**: Maximum distance between samples (most important)
- **min_samples**: Minimum samples in neighborhood
- **metric**: Distance metric ('euclidean', 'manhattan', etc.)

### Advantages

- No need to specify number of clusters
- Handles clusters of arbitrary shape
- Automatically identifies noise/outliers
- Robust to outliers
- Works well with varying cluster densities
- Only two parameters to tune

### Limitations

- Sensitive to eps parameter
- Struggles with clusters of very different densities
- Not suitable for high-dimensional data (curse of dimensionality)
- Border points may be assigned to wrong cluster
- Can be slow for large datasets


## Implementation

Let's implement DBSCAN clustering.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# Import our helper functions
from src.models.unsupervised import dbscan_cluster, evaluate_clustering

print("Libraries imported successfully!")


In [ ]:
# Generate dataset with non-spherical clusters (moons)
X, y_true = make_moons(n_samples=300, noise=0.1, random_state=42)

print(f"Dataset Shape: {X.shape}")
print(f"True number of clusters: {len(np.unique(y_true))}")

# Apply DBSCAN
dbscan = DBSCAN(eps=0.3, min_samples=10)
y_pred = dbscan.fit_predict(X)

# Count clusters and noise
n_clusters = len(set(y_pred)) - (1 if -1 in y_pred else 0)
n_noise = list(y_pred).count(-1)

print(f"\nDBSCAN Results:")
print(f"  Number of clusters: {n_clusters}")
print(f"  Number of noise points: {n_noise}")
print(f"  eps: {dbscan.eps}")
print(f"  min_samples: {dbscan.min_samples}")

# Visualize
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
plt.title('True Clusters')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
unique_labels = set(y_pred)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
for k, col in zip(unique_labels, colors):
    if k == -1:
        # Noise points in black
        col = 'black'
        marker = 'x'
        label = 'Noise'
    else:
        marker = 'o'
        label = f'Cluster {k}'
    
    class_member_mask = (y_pred == k)
    xy = X[class_member_mask]
    plt.scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker, s=50, alpha=0.7, label=label)

plt.title('DBSCAN Clustering')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Finding Optimal eps Parameter

Let's use the k-distance graph to find optimal eps.


In [ ]:
# K-distance graph method to find optimal eps
min_samples = 10
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(X)
distances, indices = neighbors_fit.kneighbors(X)

# Sort distances to k-th nearest neighbor
distances = np.sort(distances, axis=0)
distances = distances[:, min_samples - 1]

# Plot k-distance graph
plt.figure(figsize=(10, 6))
plt.plot(distances)
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{min_samples}-th Nearest Neighbor Distance')
plt.title('K-Distance Graph for DBSCAN')
plt.axhline(y=0.3, color='r', linestyle='--', label='eps=0.3')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find "knee" point (elbow)
from scipy.spatial.distance import cdist
from scipy.stats import zscore

# Use z-score to find outliers in distances (potential knee)
z_scores = np.abs(zscore(distances))
knee_idx = np.argmax(z_scores > 2)  # First point significantly above mean
optimal_eps = distances[knee_idx] if knee_idx > 0 else distances[len(distances) // 4]

print(f"Suggested eps (from k-distance graph): {optimal_eps:.3f}")
print(f"Current eps: {dbscan.eps}")


## Comparing Different eps Values

Let's see how eps affects clustering results.


In [ ]:
# Test different eps values
eps_values = [0.1, 0.2, 0.3, 0.4, 0.5]
fig, axes = plt.subplots(1, len(eps_values), figsize=(20, 4))

for idx, eps in enumerate(eps_values):
    dbscan_test = DBSCAN(eps=eps, min_samples=10)
    labels_test = dbscan_test.fit_predict(X)
    
    n_clusters_test = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    n_noise_test = list(labels_test).count(-1)
    
    unique_labels = set(labels_test)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    
    for k, col in zip(unique_labels, colors):
        if k == -1:
            col = 'black'
            marker = 'x'
        else:
            marker = 'o'
        
        class_member_mask = (labels_test == k)
        xy = X[class_member_mask]
        axes[idx].scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker, s=30, alpha=0.7)
    
    axes[idx].set_title(f'eps={eps}\nClusters: {n_clusters_test}, Noise: {n_noise_test}')
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary
print("\nSummary of eps values:")
for eps in eps_values:
    dbscan_test = DBSCAN(eps=eps, min_samples=10)
    labels_test = dbscan_test.fit_predict(X)
    n_clusters_test = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    n_noise_test = list(labels_test).count(-1)
    print(f"  eps={eps}: {n_clusters_test} clusters, {n_noise_test} noise points")


## Validation & Testing

Let's validate the clustering and compare with K-Means.


In [ ]:
# Evaluate clustering (only if we have at least 2 clusters)
if n_clusters >= 2:
    evaluation = evaluate_clustering(X, y_pred, algorithm='DBSCAN')
    print("Clustering Evaluation:")
    print(f"  Silhouette Score: {evaluation['silhouette_score']:.3f}")
    print(f"  Number of clusters: {evaluation['n_clusters']}")
    print(f"  Number of noise points: {n_noise}")
else:
    print(f"Warning: Only {n_clusters} cluster(s) found. Silhouette score requires at least 2 clusters.")
    print(f"  Number of clusters: {n_clusters}")
    print(f"  Number of noise points: {n_noise}")

# Compare with K-Means
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2, random_state=42)
y_kmeans = kmeans.fit_predict(X)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)
noise_mask = (y_pred == -1)
if np.any(noise_mask):
    plt.scatter(X[noise_mask, 0], X[noise_mask, 1], c='black', marker='x', s=50, label='Noise')
plt.title('DBSCAN (handles non-spherical)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=y_kmeans, cmap='viridis', s=50, alpha=0.7)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='red', marker='x', s=200, linewidths=3, label='Centroids')
plt.title('K-Means (assumes spherical)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nComparison:")
print(f"  DBSCAN: {n_clusters} clusters, {n_noise} noise points")
print(f"  K-Means: 2 clusters (forced), 0 noise points")
print("  Note: DBSCAN can handle non-spherical clusters, K-Means cannot")

# Assertions
assert n_clusters > 0, "Should find at least one cluster"
print("\n✓ Validation checks passed")


## Real-World Application

Let's apply DBSCAN to different types of datasets.


In [ ]:
# Test on different datasets
datasets = {
    'Circles': make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42),
    'Blobs': make_blobs(n_samples=300, centers=4, n_features=2, random_state=42, cluster_std=0.60)
}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, (name, (X_data, y_true_data)) in enumerate(datasets.items()):
    # Apply DBSCAN
    dbscan_data = DBSCAN(eps=0.3, min_samples=10)
    y_pred_data = dbscan_data.fit_predict(X_data)
    
    n_clusters_data = len(set(y_pred_data)) - (1 if -1 in y_pred_data else 0)
    n_noise_data = list(y_pred_data).count(-1)
    
    # Plot true labels
    axes[idx, 0].scatter(X_data[:, 0], X_data[:, 1], c=y_true_data, cmap='viridis', s=50, alpha=0.7)
    axes[idx, 0].set_title(f'{name} - True Labels')
    axes[idx, 0].set_xlabel('Feature 1')
    axes[idx, 0].set_ylabel('Feature 2')
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Plot DBSCAN results
    unique_labels = set(y_pred_data)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    for k, col in zip(unique_labels, colors):
        if k == -1:
            col = 'black'
            marker = 'x'
        else:
            marker = 'o'
        
        class_member_mask = (y_pred_data == k)
        xy = X_data[class_member_mask]
        axes[idx, 1].scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker, s=50, alpha=0.7)
    
    axes[idx, 1].set_title(f'{name} - DBSCAN\nClusters: {n_clusters_data}, Noise: {n_noise_data}')
    axes[idx, 1].set_xlabel('Feature 1')
    axes[idx, 1].set_ylabel('Feature 2')
    axes[idx, 1].grid(True, alpha=0.3)
    
    print(f"{name}: {n_clusters_data} clusters, {n_noise_data} noise points")

plt.tight_layout()
plt.show()


## Summary & Key Takeaways

### Key Concepts Learned

1. **DBSCAN Basics**
   - Density-based clustering algorithm
   - Groups points that are closely packed
   - Automatically identifies noise/outliers
   - No need to specify number of clusters

2. **Point Types**
   - **Core Point**: Has enough neighbors within eps
   - **Border Point**: Reachable from core but not core itself
   - **Noise Point**: Neither core nor border (outlier)

3. **Key Parameters**
   - **eps**: Maximum distance for neighbors (most critical)
   - **min_samples**: Minimum neighbors to be core point
   - Use k-distance graph to find optimal eps

4. **Best Practices**
   - Use k-distance graph to choose eps
   - Start with min_samples = 2 * dimensions
   - Scale features before clustering
   - Visualize results to validate
   - Handle noise points appropriately

### When to Use DBSCAN

✅ **Good for:**
- Unknown number of clusters
- Non-spherical cluster shapes
- Need to identify outliers/noise
- Clusters of varying density
- Spatial/geographic data
- When K-Means fails (non-spherical)

❌ **Not ideal for:**
- High-dimensional data (curse of dimensionality)
- Clusters with very different densities
- When all points should be clustered (no noise)
- Very large datasets (can be slow)
- When clusters are well-separated spheres (K-Means better)

### Next Steps

- Try **HDBSCAN** (Hierarchical DBSCAN) for varying densities
- Compare with **K-Means** and **Hierarchical Clustering**
- Use **OPTICS** for automatic parameter selection
- Apply to **anomaly detection** problems
- Use for **image segmentation** tasks
